Here are the list of documents need to process and chunk.

| Document Category | Target Manual Title                                                       | Why need it                                                   |
|-------------------|---------------------------------------------------------------------------|---------------------------------------------------------------|
| Bridges           | VicRoads Road Structures Inspection Manual                                | Defining condition states for bridge components.              |
| Roads/Pavement    | VicRoads Technical Bulletin TB 50: Guide to Surface Inspection Rating     | Focuses on asphalt and sprayed seal surfaces.                 |
| Repair Standards  | VicRoads Section 687: Repair of Concrete Cracks                           | Provide engineering methods the LLM should recommend.         |
| Repair Standards  | VicRoads Section 689: Cementitious Patch Repair                           | Useful for larger spalling or severe crack repairs.           |
| General           | Austroads Guide to Bridge Technology (Part 7: Maintenance and Management) | Used across Australia to provide general maintenance context. |

In [27]:
import fitz
from marker.converters.pdf import PdfConverter
from marker.models import create_model_dict
from marker.config.parser import ConfigParser
import pathlib
import re
import io
import gc
from docling.document_converter import DocumentConverter
from docling.datamodel.document import DocumentStream

# Convert documents to markdowns

## Roads/Pavement

There is a table of content in this pdf, need to be removed.

In [28]:
# Check total pages
pdf = fitz.open("raw_documents/Technical Bulletin TB 50  Guide to Surface Inspection Rating.pdf")
print(pdf.page_count)

62


Table of content ends at page 6

In [ ]:
config = {"page_range": f"6-{pdf.page_count - 1}"}
config_parser = ConfigParser(config)

converter = PdfConverter(
    artifact_dict=create_model_dict(),
    config=config_parser.generate_config_dict()
)
rendered = converter("raw_documents/Technical Bulletin TB 50  Guide to Surface Inspection Rating.pdf")
markdown_text = rendered.markdown

In [ ]:
print(markdown_text[:10000])

### 1.1 General

Surface Inspection Rating (SIR) is a standardised system for assessment of the pavement surface condition for sprayed seal and asphalt wearing courses based on visual inspection.

This document provides a guide to the use of SIR for evaluation of the condition of sprayed seal surfaces and asphalt wearing course surfaces by means of a combination of a visual "walkover" inspection, carried out at strategic locations on the pavement surface, and other information available on performance and serviceability.

SIR is to be adopted by all personnel responsible for undertaking surface inspections, as stipulated in the Pavement Conditions Inspections Policy. This will assist in ensuring a more consistent and repeatable approach to the visual assessment of sprayed seal and asphalt wearing course surface conditions.

Detailed definitions, conventions, information and photographs are provided to assist with the SIR assessment. Separate visual assessment criteria are provided for 

The markdown seems good. One slight problem with the `#`, the large headers sometime are misplaced. This is due to the layout of the document. That I will manual fix and save in `processed_documents` folder. Later I will use layout awareness chunking for this document.

In [ ]:
# Save as markdown
pathlib.Path("raw_documents/Technical Bulletin TB 50  Guide to Surface Inspection Rating.md").write_bytes(markdown_text.encode())
print("Done saving!")

Done saving!


## Bridges

In [ ]:
pdf_path = "raw_documents/VicRoads Road Structures Inspection Manual.pdf"
pdf = fitz.open(pdf_path)
print(pdf.page_count)

36


Table of contend ends at page 3. There are many pictures in this documents, I think I will just leave the images and not use that for RAG for now.

In [ ]:
config = {"page_range": f"3-{pdf.page_count - 1}"}
config_parser = ConfigParser(config)

converter = PdfConverter(
    artifact_dict=create_model_dict(),
    config=config_parser.generate_config_dict()
)
rendered = converter(pdf_path)
markdown_text = rendered.markdown

Recognizing tables: 100%|██████████| 1/1 [00:00<00:00,  1.51it/s]
Detecting bboxes: 0it [00:00, ?it/s]


In [ ]:
print(markdown_text[:2000])

# 60P - Pipe culverts – precast concrete Units

This component includes all precast concrete pipes and includes the jointing arrangements between them.

**m<sup>2</sup>** of exposed surface area

Note, this component does not include propping. If culvert is propped, refer to Component No. 18.

# **Condition state 1 -Description Photo**

- The component may show only minor superficial cracking of no consequence.
- The line and invert of the pipe is straight with no water being retained within the pipe.

![](_page_3_Picture_9.jpeg)

Precast concrete pipes in good condition

- The component may show minor cracking or spalling due to corroding reinforcement in isolated areas.
- The line of the pipe is straight but minor settlement of some units may be allowing a minor pool of water to be retained in the pipe.

![](_page_3_Picture_15.jpeg)

Minor cracking along pipe haunch line

- The component may show moderate cracking or spalling due to corroding reinforcement in isolated areas.
- The li

This method doesn't work on this document. It puts so many header `#` and the `Condition state` is missing for a few of them. I will try another method. I will detect large blue headings from pdf.

In [ ]:
# Extract lines rendered as large blue text in the PDF
blue_heading_texts = set()

for page in pdf.pages(3):
    for block in page.get_text("dict")["blocks"]:
        if block["type"] != 0:
            continue
        for line in block["lines"]:
            line_text = ""
            has_large_blue = False
            for span in line["spans"]:
                size = span["size"]
                color = span["color"]
                r = (color >> 16) & 0xFF
                g = (color >> 8) & 0xFF
                b = color & 0xFF
                # Large blue: font size >= 11, blue channel dominant
                if size >= 11 and b >= 100 and b > r and b > g:
                    has_large_blue = True
                line_text += span["text"]
            if has_large_blue and line_text.strip():
                blue_heading_texts.add(line_text.strip())

print(f"Found {len(blue_heading_texts)} large blue heading lines:")
for t in sorted(blue_heading_texts):
    print(" ", repr(t))

Found 25 large blue heading lines:
  '60O - Pipe culverts – other'
  '60P - Pipe culverts – precast concrete'
  '60S - Pipe culverts – steel'
  '61C - Box culverts – cast in-situ concrete'
  '61O - Box culverts – masonry'
  '61P - Box culverts – precast concrete'
  '62C - Arch culverts – cast in-situ concrete'
  '62O - Arch culverts – masonry'
  '62O - Arch culverts – steel'
  '62P - Arch culverts – precast concrete'
  '63C - Headwalls/wingwalls – cast in-situ concrete'
  '63O - Headwalls/wingwalls – other'
  '63P - Headwalls/wingwalls - precast concrete'
  '64C - Culvert base slab / Steel pipe invert – concrete'
  '64O - Culvert base slab / Steel pipe invert – other'
  '65 - Waterway'
  '66 - Channel protection'
  '67 – Traffic barrier'
  'Each'
  'Linear'
  'area'
  'exposed'
  'm2 of'
  'metres'
  'surface'


There are a few false positives like 'Each', 'Linear' ... and fitz uses `–` but docling uses `-`. I will be using docling for this markdown extraction. I will filter changing `–` to `-`, and remove string less than 8 characters.

In [ ]:
blue_heading_texts = {bh.replace('–', '-').replace('—', '-').lower() for bh in blue_heading_texts if len(bh.strip()) > 8}
blue_heading_texts

{'60o - pipe culverts - other',
 '60p - pipe culverts - precast concrete',
 '60s - pipe culverts - steel',
 '61c - box culverts - cast in-situ concrete',
 '61o - box culverts - masonry',
 '61p - box culverts - precast concrete',
 '62c - arch culverts - cast in-situ concrete',
 '62o - arch culverts - masonry',
 '62o - arch culverts - steel',
 '62p - arch culverts - precast concrete',
 '63c - headwalls/wingwalls - cast in-situ concrete',
 '63o - headwalls/wingwalls - other',
 '63p - headwalls/wingwalls - precast concrete',
 '64c - culvert base slab / steel pipe invert - concrete',
 '64o - culvert base slab / steel pipe invert - other',
 '65 - waterway',
 '66 - channel protection',
 '67 - traffic barrier'}

In [ ]:
# Convert PDF to markdown using docling, skipping first 3 pages
subset_pdf = fitz.open()
subset_pdf.insert_pdf(pdf, from_page=3, to_page=pdf.page_count - 1)
buf = io.BytesIO(subset_pdf.tobytes())

doc_converter = DocumentConverter()
result = doc_converter.convert(DocumentStream(name=pdf_path, stream=buf))
raw_markdown = result.document.export_to_markdown()

# Fix headings
# - Large blue text = #
# - Condition State =  ##
# - Everything else =  regular text
def fix_headings(md):
    def classify(line):
        text = re.sub(r"^#+\s*", "", line).strip()
        t = text.lower().replace('–', '-').replace('—', '-')
        if any(t in b or b in t for b in blue_heading_texts): return f"# {text}"
        if re.match(r"condition\s+state", text, re.IGNORECASE):        return f"## {text}"
        return text
    return "\n".join(classify(l) if l.startswith("#") else l for l in md.splitlines())

markdown_text = fix_headings(raw_markdown)
print(markdown_text[:5000])

[INFO] 2026-03-12 00:37:22,620 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-03-12 00:37:22,623 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\s223128143\AppData\Local\miniconda3\envs\myenv\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-03-12 00:37:22,623 [RapidOCR] main.py:53: Using C:\Users\s223128143\AppData\Local\miniconda3\envs\myenv\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-03-12 00:37:22,682 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-03-12 00:37:22,682 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\s223128143\AppData\Local\miniconda3\envs\myenv\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-03-12 00:37:22,684 [RapidOCR] main.py:53: Using C:\Users\s223128143\AppData\Local\miniconda3\envs\myenv\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-03-12 00:37:22,714 [RapidOCR] base

# 60P - Pipe culverts - precast concrete

This component includes all precast concrete pipes and includes the jointing arrangements between them.

Note, this component does not include propping. If culvert is propped, refer to Component No. 18.

## Condition state 1 -Description

-  The component may show only minor superficial cracking of no consequence.
-  The line and invert of the pipe is straight with no water being retained within the pipe.

## Condition state 2 -Description

-  The component may show minor cracking or spalling due to corroding reinforcement in isolated areas.
-  The line of the pipe is straight but minor settlement of some units may be allowing a minor pool of water to be retained in the pipe.

Photo

Precast concrete pipes in good condition

<!-- image -->

Photo

Minor cracking along pipe haunch line

<!-- image -->

Units m 2 of exposed surface area

## Condition state 3 -Description

-  The component may show moderate cracking or spalling due to corrodi

In [ ]:
pathlib.Path("raw_documents/VicRoads Road Structures Inspection Manual.md").write_bytes(markdown_text.encode())
print("Done saving!")

Done saving!


## Repair Standards

In [ ]:
pdf_path = "raw_documents/VicRoads Section 687 Repair of Concrete Cracks.pdf"
pdf = fitz.open(pdf_path)
print(pdf.page_count)

8


No header on this one.

In [ ]:
doc_converter = DocumentConverter()
result = doc_converter.convert(pdf_path)
raw_markdown = result.document.export_to_markdown()

[INFO] 2026-03-13 20:46:09,091 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-03-13 20:46:09,124 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\s223128143\AppData\Local\miniconda3\envs\myenv\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-03-13 20:46:09,125 [RapidOCR] main.py:53: Using C:\Users\s223128143\AppData\Local\miniconda3\envs\myenv\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-03-13 20:46:09,223 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-03-13 20:46:09,241 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\s223128143\AppData\Local\miniconda3\envs\myenv\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-03-13 20:46:09,243 [RapidOCR] main.py:53: Using C:\Users\s223128143\AppData\Local\miniconda3\envs\myenv\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-03-13 20:46:09,276 [RapidOCR] base

In [ ]:
print(raw_markdown[:5000])

## SECTION 687  -  REPAIR OF CONCRETE CRACKS

##This section cross-references  Sections 168, 176, 686 and 689.

If any of the above sections are relevant,  they should be included in the specification.

If any of the above sections are not included in the specification, all references  to those sections should be struck out, ensuring that the remaining text is still coherent:

## 687.01 GENERAL

This section specifies the requirements  for the supply and quality of materials,  surface preparation, application, relevant testing and acceptance  criteria for the repair of cracks  in concrete.

Repair of cracks shall  not be undertaken  unless the cracked concrete structure has been assessed and the influence  of cracks on load bearing capacity,  serviceability  and durability  has been evaluated by the Contractor,  and reviewed by the Superintendent.

A crack repair method  shall be selected based on the assessment  of the cause(s)  of the crack,  crack width, the moisture condition of th

This document doesn't seem to have any complicated headers.

In [ ]:
pathlib.Path("raw_documents/VicRoads Section 687 Repair of Concrete Cracks.md").write_bytes(raw_markdown.encode())
print("Done saving!")

Done saving!


Second repair standard documents.

In [ ]:
pdf_path = "raw_documents/VicRoads Section 689 Cementitious Patch Repair.pdf"
pdf = fitz.open(pdf_path)
print(pdf.page_count)

9


In [ ]:
doc_converter = DocumentConverter()
result = doc_converter.convert(pdf_path)
raw_markdown = result.document.export_to_markdown()

[INFO] 2026-03-13 20:50:23,331 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-03-13 20:50:23,336 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\s223128143\AppData\Local\miniconda3\envs\myenv\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-03-13 20:50:23,336 [RapidOCR] main.py:53: Using C:\Users\s223128143\AppData\Local\miniconda3\envs\myenv\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-03-13 20:50:23,393 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-03-13 20:50:23,395 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\s223128143\AppData\Local\miniconda3\envs\myenv\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-03-13 20:50:23,396 [RapidOCR] main.py:53: Using C:\Users\s223128143\AppData\Local\miniconda3\envs\myenv\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-03-13 20:50:23,445 [RapidOCR] base

In [ ]:
print(raw_markdown[:5000])

## SECTION 689  -  CEMENTITIOUS PATCH REPAIR OF CONCRETE

##This  section cross-references Sections  168, 176, 611, 685, 686 and 687. If any of the above sections are relevant, they should be included  in the specification.

If any of the above sections are not included in the specification, all references to those sections should  be struck out, ensuring  that the remaining  text is still  coherent:

## 689.01 GENERAL

This section specifies the requirements  for  the supply  of  materials,  surface  preparation, application, relevant inspection and testing and acceptance criteria for the patch repair of concrete structures  using  cementitious  repair materials.

## 689.02 TYPES AND SELECTION OF PATCH REPAIR METHODS

This section includes the following types of patch repair of concrete structures using cementitious repair materials:

- corrosion deteriorated concrete repair;
- non-corrosion  deteriorated concrete repair; and
- filling  of blowholes  and surface imperfections.

Repair

In [ ]:
pathlib.Path("raw_documents/VicRoads Section 689 Cementitious Patch Repair.md").write_bytes(raw_markdown.encode())
print("Done saving!")

Done saving!


## General

In [ ]:
pdf_path = "raw_documents/Austroads Guide to Bridge Technology Part 7.pdf"
pdf = fitz.open(pdf_path)
print(pdf.page_count)

267


We will skip the first 10 pages. Since there are many pages, we will convert to markdown in batches. 

In [33]:
start_page = 10 # skip first 10 pages 
end_page = fitz.open(pdf_path).page_count - 1

batch_size = 5
force_skip_pages = set()

markdown_parts = []
failed_pages = []

doc_converter = DocumentConverter()

src_pdf = fitz.open(pdf_path)

for batch_start in range(start_page, end_page + 1, batch_size):
    batch_end = min(batch_start + batch_size - 1, end_page)

    # Skip whole batch only if every page is forced-skip
    batch_pages = set(range(batch_start, batch_end + 1))
    if batch_pages.issubset(force_skip_pages):
        continue

    subset_pdf = fitz.open()
    try:
        subset_pdf.insert_pdf(src_pdf, from_page=batch_start, to_page=batch_end)
        buf = io.BytesIO(subset_pdf.tobytes())

        result = doc_converter.convert(
            DocumentStream(name=f"{pdf_path}#pages={batch_start}-{batch_end}", stream=buf)
        )
        md = result.document.export_to_markdown()
        markdown_parts.append(md)

        print(f"OK pages {batch_start}-{batch_end}")

    except Exception as e:
        # Fall back to single-page attempts for this batch
        print(f"Batch failed {batch_start}-{batch_end}: {e}")
        for p in range(batch_start, batch_end + 1):
            if p in force_skip_pages:
                continue
            single_pdf = fitz.open()
            try:
                single_pdf.insert_pdf(src_pdf, from_page=p, to_page=p)
                single_buf = io.BytesIO(single_pdf.tobytes())
                single_result = doc_converter.convert(
                    DocumentStream(name=f"{pdf_path}#page={p}", stream=single_buf)
                )
                markdown_parts.append(single_result.document.export_to_markdown())
                print(f"OK page {p}")
            except Exception as e2:
                failed_pages.append((p, str(e2)))
                print(f"FAILED page {p}: {e2}")
            finally:
                single_pdf.close()
                gc.collect()
    finally:
        subset_pdf.close()
        gc.collect()

src_pdf.close()

raw_markdown = "\n\n".join(markdown_parts)
out_path = pathlib.Path("raw_documents/Austroads Guide to Bridge Technology Part 7.md")
out_path.write_bytes(raw_markdown.encode("utf-8"))

print(f"Saved markdown: {out_path}")
print(f"Failed pages count: {len(failed_pages)}")
if failed_pages:
    print("Failed pages:")
    for p, err in failed_pages:
        print(f"  - page {p}: {err}")

[INFO] 2026-03-13 21:35:22,469 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-03-13 21:35:22,473 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\s223128143\AppData\Local\miniconda3\envs\myenv\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-03-13 21:35:22,473 [RapidOCR] main.py:53: Using C:\Users\s223128143\AppData\Local\miniconda3\envs\myenv\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-03-13 21:35:22,537 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-03-13 21:35:22,538 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\s223128143\AppData\Local\miniconda3\envs\myenv\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-03-13 21:35:22,539 [RapidOCR] main.py:53: Using C:\Users\s223128143\AppData\Local\miniconda3\envs\myenv\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-03-13 21:35:22,575 [RapidOCR] base

OK pages 10-14
OK pages 15-19
OK pages 20-24
OK pages 25-29
OK pages 30-34
OK pages 35-39
OK pages 40-44
OK pages 45-49
OK pages 50-54
OK pages 55-59
OK pages 60-64
OK pages 65-69
OK pages 70-74
OK pages 75-79
OK pages 80-84


RapidOCR returned empty result!


OK pages 85-89


RapidOCR returned empty result!
RapidOCR returned empty result!
[WARNING] 2026-03-13 21:36:57,551 [RapidOCR] main.py:125: The text detection result is empty
RapidOCR returned empty result!


...

OK pages 210-214
OK pages 215-219
OK pages 220-224
OK pages 225-229
OK pages 230-234
OK pages 235-239
OK pages 240-244
OK pages 245-249


RapidOCR returned empty result!
RapidOCR returned empty result!
RapidOCR returned empty result!
RapidOCR returned empty result!
[WARNING] 2026-03-13 21:38:53,731 [RapidOCR] main.py:125: The text detection result is empty
RapidOCR returned empty result!


OK pages 250-254


RapidOCR returned empty result!
RapidOCR returned empty result!
RapidOCR returned empty result!
RapidOCR returned empty result!
RapidOCR returned empty result!
RapidOCR returned empty result!
RapidOCR returned empty result!
RapidOCR returned empty result!
RapidOCR returned empty result!
RapidOCR returned empty result!


OK pages 255-259
OK pages 260-264
OK pages 265-266
Saved markdown: raw_documents\Austroads Guide to Bridge Technology Part 7.md
Failed pages count: 0


In [34]:
print(raw_markdown[:5000])

## 1. Introduction

## 1.1 Scope

Part 7 of the Austroads Guide to Bridge Technology (AGBT) provides guidance on the maintenance and management of existing bridges and addresses topics such as:

- bridge inspection
- load rating and monitoring
- maintenance, repair and strengthening
- bridge preservation.

Part 7 provides documentation of the technical issues and practices relevant to the above topics and is intended to present good practice in the technical aspects of bridge management consistent with Integrated Asset Management Guidelines for Road Networks (Austroads 2002). Part 7 can be used to complement Guide to Asset Management Technical Information Part 13: Structures Asset Management (Austroads 2018), which provides guidance on the establishment and maintenance of structure asset inventories and the monitoring of asset performance.

The focus of Part 7 is on the practical and technical procedures and strategies available for managing the existing bridges in a network and mainta

The extraction is good but I now need to normalize the heading. Something like this
```
1. Title -> #
1.1 Title -> ##
1.1.1 Title -> ###
etc.
```

In [35]:
def normalize_numbered_headings(md: str) -> str:
    lines = md.splitlines()
    out = []

    # Matches markdown headings like:
    # ## 1. Intro
    # ## 1.1 Scope
    # ## 1.1.1 Details
    numbered_heading_re = re.compile(
        r'^\s*#{1,6}\s+(\d+(?:\.\d+)*\.?)\s+(.+?)\s*$'
    )

    for line in lines:
        m = numbered_heading_re.match(line)
        if not m:
            out.append(line)
            continue

        number_token = m.group(1)          # e.g. 1. or 1.1 or 2.5.1
        title = m.group(2).strip()

        # Depth based on numeric segments: 1 -> 1, 1.1 -> 2, 1.1.1 -> 3
        normalized_num = number_token.rstrip(".")
        depth = normalized_num.count(".") + 1
        depth = max(1, min(depth, 6))

        out.append(f'{"#" * depth} {number_token} {title}')

    return "\n".join(out)

In [38]:
processed_markdown = normalize_numbered_headings(raw_markdown)
out_path = pathlib.Path("raw_documents/Austroads Guide to Bridge Technology Part 7.md")
out_path.write_bytes(processed_markdown.encode("utf-8"))
print(processed_markdown[:5000])

# 1. Introduction

## 1.1 Scope

Part 7 of the Austroads Guide to Bridge Technology (AGBT) provides guidance on the maintenance and management of existing bridges and addresses topics such as:

- bridge inspection
- load rating and monitoring
- maintenance, repair and strengthening
- bridge preservation.

Part 7 provides documentation of the technical issues and practices relevant to the above topics and is intended to present good practice in the technical aspects of bridge management consistent with Integrated Asset Management Guidelines for Road Networks (Austroads 2002). Part 7 can be used to complement Guide to Asset Management Technical Information Part 13: Structures Asset Management (Austroads 2018), which provides guidance on the establishment and maintenance of structure asset inventories and the monitoring of asset performance.

The focus of Part 7 is on the practical and technical procedures and strategies available for managing the existing bridges in a network and maintai